In [ ]:
groq_api_key = "your_api_key"

In [4]:
!pip install -q PyPDF2 sentence-transformers faiss-cpu groq scikit-learn "numpy==2.0.2"

In [ ]:
import numpy as np
import faiss
import PyPDF2
from groq import Groq
from sentence_transformers import SentenceTransformer, CrossEncoder



from google.colab import userdata
client = Groq(api_key='your_api_key')

In [5]:
print("Loading Bi-Encoder (for fast search) and Cross-Encoder (for precision)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

Loading Bi-Encoder (for fast search) and Cross-Encoder (for precision)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [9]:
# you can either do chunking with overlap or else,
# use a transformer llm to break (to chunks) in cases of particular topic endings . (advance)

In [10]:
pdf_path = "/content/deep-learning-material-dept-ece-ase-blr-1.pdf"

def extract_text_from_pdf(file_path):
    """Raw extraction of text from the PDF pages."""
    text = ""
    with open(file_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        for page_num in range(len(reader.pages)):
            page_text = reader.pages[page_num].extract_text()
            if page_text:
                text += page_text + "\n"
    return text

print(f"Extracting text from {pdf_path}...")
raw_text = extract_text_from_pdf(pdf_path)

Extracting text from /content/deep-learning-material-dept-ece-ase-blr-1.pdf...


In [11]:
def chunk_text(text, chunk_size=250, overlap=50):

    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk not in chunks:
            chunks.append(chunk)
    return chunks

docs = chunk_text(raw_text)
print(f"Total chunks created: {len(docs)}")

Total chunks created: 590


# 🔍 Vector Search / FAISS Notes

## Brute Force Method
- Compare query vector with **ALL stored vectors**
- Works across all **dimensions (columns)**
- ✅ Very accurate  
- ❌ SLOW for large datasets  

---

## FAISS
- Library for fast similarity search on **vectors (embeddings)**
- Can handle **very large-scale data efficiently**
- Provides multiple **indexing architectures**

---

## Architectures

### 1. Flat (Brute Force)
- Exact search  
- Checks every vector  
- ✅ Accurate  
- ❌ Slow  

### 2. Graph-Based (HNSW) 🚀
- Stores vectors as a **graph (neighbors connected)**
- Searches by navigating through **nearest neighbors**
- ✅ VERY FAST  
- ✅ Scalable  
- ⭐ Best in practice  

---

## Finding Closest Chunks
- Uses **distance metrics** to measure similarity  

---

## Euclidean Distance (L2)
- Measures **straight-line distance** between vectors  
- Smaller distance → more similar  
- Larger distance → less similar  

In [13]:
print("Generating embeddings for all chunks ")
doc_embeddings = embedder.encode(docs)

# Build the FAISS Index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
#brute force method, dim :- columns, faiss has multi architecture u can store data , sabse best is the graoh architecture
# graph architecture is very fast and best.
# closest chunks is retrieved by ---- equilidian similarity , this L2 is doing this
index.add(np.array(doc_embeddings).astype("float32"))
print(f"Successfully added {index.ntotal} vectors to the FAISS index.")

Generating embeddings for all chunks 
Successfully added 590 vectors to the FAISS index.


In [14]:
def generate_with_groq(prompt):
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful teaching assistant. Answer strictly based on the provided textbook context."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        model="llama-3.1-8b-instant",
        temperature=0.1
    )
    return chat_completion.choices[0].message.content

def ask_pdf(query, initial_k=10, final_k=3):

    # Stage 1: Fast Retrieval via Euclidean distance in FAISS
    query_vector = embedder.encode([query]).astype("float32") # the embeder that u used to convert chunks u need to use the same embeder to convert the query.
    _, indices = index.search(query_vector, initial_k)
    candidate_docs = [docs[idx] for idx in indices[0]]

    # Stage 2: Reranking for high precision using Cross-Encoder
    pairs = [[query, doc] for doc in candidate_docs]
    scores = reranker.predict(pairs)

    # Sort and pick the absolute best chunks
    ranked_indices = np.argsort(scores)[::-1]
    best_docs = [candidate_docs[i] for i in ranked_indices[:final_k]]

    combined_context = "\n\n---\n\n".join(best_docs)

    # Stage 3: LLM Generation
    prompt = f"""
    Context information from the textbook is below.
    ---------------------
    {combined_context}
    ---------------------
    Given the context information, answer the question: {query}
    """

    return generate_with_groq(prompt)

In [ ]:
# this transformer converts into dense vectors ?